In [18]:
import pandas as pd

df = pd.read_csv("../dataset/fake_job_postings.csv")

print(df.shape)

(17880, 18)


In [20]:
target = "fraudulent"

X = df.drop(columns=[target])
y = df[target]

In [21]:
X = X.drop(columns=["job_id"])

In [22]:
print(df.shape)
print(X.shape)

(17880, 18)
(17880, 16)


In [23]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

In [24]:
categorical_columns = [
    "location",
    "department",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

In [25]:
numeric_columns = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

In [26]:
df[text_columns] = df[text_columns].fillna("")

In [27]:
df["combined_text"] = (
    df["title"] + " " +
    df["company_profile"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["benefits"]
)

In [28]:
df["combined_text"].head()

0    Marketing Intern We're Food52, and we've creat...
1    Customer Service - Cloud Video Production 90 S...
2    Commissioning Machinery Assistant (CMA) Valor ...
3    Account Executive - Washington DC Our passion ...
4    Bill Review Manager SpotSource Solutions LLC i...
Name: combined_text, dtype: str

In [29]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [30]:
df["clean_text"] = df["combined_text"].apply(clean_text)

In [31]:
df[["combined_text", "clean_text"]].head()

,combined_text,clean_text
0,"Marketing Intern We're Food52, and we've creat...",marketing intern we re food and we ve created ...
1,Customer Service - Cloud Video Production 90 S...,customer service cloud video production second...
2,Commissioning Machinery Assistant (CMA) Valor ...,commissioning machinery assistant cma valor se...
3,Account Executive - Washington DC Our passion ...,account executive washington dc our passion fo...
4,Bill Review Manager SpotSource Solutions LLC i...,bill review manager spotsource solutions llc i...


In [32]:
categorical_columns = [
    "location",
    "department",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

In [33]:
df[categorical_columns] = df[categorical_columns].fillna("Unknown")

In [34]:
df[categorical_columns].isnull().sum()

location               0
department             0
employment_type        0
required_experience    0
required_education     0
industry               0
function               0
dtype: int64

In [35]:
features = df[
    [
        "clean_text",
        "location",
        "department",
        "employment_type",
        "required_experience",
        "required_education",
        "industry",
        "function",
        "telecommuting",
        "has_company_logo",
        "has_questions"
    ]
]

target = df["fraudulent"]

In [36]:
print("Features:", features.shape)
print("Target:", target.shape)

Features: (17880, 11)
Target: (17880,)


In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=42,
    stratify=target
)

In [38]:
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (14304, 11)
Testing: (3576, 11)


In [39]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training target distribution:
fraudulent
0    13611
1      693
Name: count, dtype: int64

Testing target distribution:
fraudulent
0    3403
1     173
Name: count, dtype: int64


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

In [41]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore"
)

In [42]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            tfidf,
            "clean_text"
        ),
        (
            "categorical",
            encoder,
            categorical_columns
        ),
        (
            "numeric",
            "passthrough",
            numeric_columns
        )
    ]
)

In [43]:
X_train_processed = preprocessor.fit_transform(X_train)

In [44]:
X_test_processed = preprocessor.transform(X_test)

In [45]:
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (14304, 14114)
Processed testing shape: (3576, 14114)


In [46]:
import joblib

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [47]:
print("========== PREPROCESSING SUMMARY ==========")
print("Original dataset:", df.shape)
print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)
print("Processed training:", X_train_processed.shape)
print("Processed testing:", X_test_processed.shape)

========== PREPROCESSING SUMMARY ==========
Original dataset: (17880, 20)
Training samples: (14304, 11)
Testing samples: (3576, 11)
Processed training: (14304, 14114)
Processed testing: (3576, 14114)
